In [254]:
#imports
import re
from collections import Counter
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence

In [255]:
# config
DATA_PATH = '..\\..\\Data\\cellula_toxic_data.csv'

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
torch.manual_seed(SEED)

MODEL_TYPE = "lstm"
EMBED_DIM = 128
HIDDEN_DIM = 32
NUM_LAYERS = 1
BIDIRECTIONAL = False
DROPOUT = 0.3
BATCH_SIZE = 32
EPOCHS = 15
LR = 1e-3
MIN_FREQ = 1
PAD_TOKEN, UNK_TOKEN = "<pad>", "<unk>"

Pipeline:
1. Load data
2. Build a raw Dataset (query, label)
3. random_split into train/val/test (index-level split -> no leakage)
4. Build vocabulary from the TRAIN split
5. Wrap numericalization using the shared vocab (applied to all splits)
6. DataLoader with padding
7. Model: Embedding -> (RNN or LSTM) -> Linear head
8. Train loop
9. Evaluate (accuracy + per-class F1, Note: classes are imbalanced)

# 1 Load data

In [256]:
df = pd.read_csv(DATA_PATH)

# 2 Build raw Dataset

In [257]:
texts = df["query"].tolist()
label_strings = df["Toxic Category"].tolist()

classes = sorted(set(label_strings))
label2idx = {c: i for i, c in enumerate(classes)}
idx2label = {i: c for c, i in label2idx.items()}
labels = [label2idx[l] for l in label_strings]

print(f"Classes: {classes}")

Classes: ['Child Sexual Exploitation', 'Elections', 'Non-Violent Crimes', 'Safe', 'Sex-Related Crimes', 'Suicide & Self-Harm', 'Unknown S-Type', 'Violent Crimes', 'unsafe']


In [258]:
class RawTextDataset(Dataset):
  def __init__(self, texts, labels):
    self.texts = texts
    self.labels = labels

  def __len__(self):
    return len(self.texts)

  def __getitem__(self, idx):
    return self.texts[idx], self.labels[idx]

# 3 Split

In [259]:
full_dataset = RawTextDataset(texts, labels)

# Split the dataset into training, validation, and test sets
n = len(full_dataset)
n_train = int(0.7 * n)
n_val = int(0.15 * n)
n_test = n - n_train - n_val

train_raw, val_raw, test_raw = random_split(
    full_dataset,
    [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(SEED),
)

# 4 Build vocab from train

In [260]:
TOKEN_RE = re.compile(r"[A-Za-z']+|\d+")
def tokenize(text: str):
  return TOKEN_RE.findall(text.lower())

In [261]:
def build_vocab(subset, min_freq=1):
  counter = Counter()
  for text, _ in subset:
    counter.update(tokenize(text))

  vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
  for token, freq in counter.items():
    if freq >= min_freq:
      vocab[token] = len(vocab)
  return vocab


vocab = build_vocab(train_raw, min_freq=MIN_FREQ)
print(f"Vocab size: {len(vocab)}")


def numericalize(text: str):
  ids = [vocab.get(tok, vocab[UNK_TOKEN]) for tok in tokenize(text)]
  if not ids:
    ids = [vocab[UNK_TOKEN]]
  return torch.tensor(ids, dtype=torch.long)

Vocab size: 3634


# 5 Tokenized Dataset

In [262]:
class TokenizedDataset(Dataset):
  def __init__(self, subset):
    self.subset = subset

  def __len__(self):
    return len(self.subset)

  def __getitem__(self, idx):
    text, label = self.subset[idx]
    return numericalize(text), torch.tensor(label, dtype=torch.long)


train_ds = TokenizedDataset(train_raw)
val_ds = TokenizedDataset(val_raw)
test_ds = TokenizedDataset(test_raw)

# 6 DataLoader

In [263]:
def collate_fn(batch):
  seqs, labels = zip(*batch)
  lengths = torch.tensor([len(s) for s in seqs], dtype=torch.long)
  padded = pad_sequence(seqs, batch_first=True, padding_value=vocab[PAD_TOKEN])
  labels = torch.stack(labels)
  return padded, lengths, labels


train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

# 7 Model

In [264]:
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes,
                num_layers=1, bidirectional=False, dropout=0.0,
                pad_idx=0, cell_type="lstm"):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)

        rnn_cls = nn.LSTM if cell_type == "lstm" else nn.RNN
        self.rnn = rnn_cls(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        directions = 2 if bidirectional else 1
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * directions, num_classes)

    def forward(self, x, lengths):
        embedded = self.embedding(x)  # (B, T, E)

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )

        if isinstance(self.rnn, nn.LSTM):
            _, (hidden, _) = self.rnn(packed)
        else:
            _, hidden = self.rnn(packed)

        # hidden: (num_layers * num_directions, B, hidden_dim)
        if self.rnn.bidirectional:
            last_hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        else:
            last_hidden = hidden[-1]

        out = self.dropout(last_hidden)
        return self.fc(out)

In [265]:
model = RNNClassifier(
    vocab_size=len(vocab),
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=len(classes),
    num_layers=NUM_LAYERS,
    bidirectional=BIDIRECTIONAL,
    dropout=DROPOUT,
    pad_idx=vocab[PAD_TOKEN],
    cell_type=MODEL_TYPE,
).to(DEVICE)

# 8 Train

In [266]:
criterion = nn.NLLLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)


def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with torch.set_grad_enabled(train):
        for x, lengths, y in loader:
            x, lengths, y = x.to(DEVICE), lengths, y.to(DEVICE)

            if train:
                optimizer.zero_grad()

            logits = model(x, lengths)
            loss = criterion(logits, y)

            if train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * x.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            total += x.size(0)

    return total_loss / total, correct / total

In [267]:
for epoch in range(1, EPOCHS + 1):
  train_loss, train_acc = run_epoch(train_loader, train=True)
  val_loss, val_acc = run_epoch(val_loader, train=False)
  print(f"Epoch {epoch:2d} | train_loss={train_loss:.4f} train_acc={train_acc:.3f} "f"| val_loss={val_loss:.4f} val_acc={val_acc:.3f}")

Epoch  1 | train_loss=-0.4450 train_acc=0.367 | val_loss=-0.9443 val_acc=0.524
Epoch  2 | train_loss=-2.7655 train_acc=0.552 | val_loss=-4.8025 val_acc=0.558
Epoch  3 | train_loss=-6.2232 train_acc=0.520 | val_loss=-7.6333 val_acc=0.549
Epoch  4 | train_loss=-8.8275 train_acc=0.532 | val_loss=-10.0228 val_acc=0.500
Epoch  5 | train_loss=-11.0088 train_acc=0.513 | val_loss=-12.2385 val_acc=0.500
Epoch  6 | train_loss=-13.2211 train_acc=0.526 | val_loss=-14.3625 val_acc=0.498
Epoch  7 | train_loss=-15.4014 train_acc=0.541 | val_loss=-16.4351 val_acc=0.529
Epoch  8 | train_loss=-17.3611 train_acc=0.529 | val_loss=-18.4727 val_acc=0.529
Epoch  9 | train_loss=-19.4082 train_acc=0.532 | val_loss=-20.4629 val_acc=0.529
Epoch 10 | train_loss=-21.3687 train_acc=0.529 | val_loss=-22.4477 val_acc=0.493
Epoch 11 | train_loss=-23.4874 train_acc=0.519 | val_loss=-24.4251 val_acc=0.493
Epoch 12 | train_loss=-25.2846 train_acc=0.506 | val_loss=-26.3728 val_acc=0.496
Epoch 13 | train_loss=-27.3097 trai

# 9 Evaluate

In [268]:
def evaluate(loader):
  model.eval()
  all_preds, all_labels = [], []
  with torch.no_grad():
      for x, lengths, y in loader:
          x = x.to(DEVICE)
          output = model(x, lengths)
          preds = output.argmax(1).cpu()
          all_preds.extend(preds.tolist())
          all_labels.extend(y.tolist())
  return all_preds, all_labels


preds, gold = evaluate(test_loader)

def classification_report(preds, gold, idx2label):
  classes_ = sorted(idx2label.keys())
  for c in classes_:
      tp = sum(1 for p, g in zip(preds, gold) if p == c and g == c)
      fp = sum(1 for p, g in zip(preds, gold) if p == c and g != c)
      fn = sum(1 for p, g in zip(preds, gold) if p != c and g == c)
      precision = tp / (tp + fp) if (tp + fp) else 0.0
      recall = tp / (tp + fn) if (tp + fn) else 0.0
      f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
      print(f"{idx2label[c]:20s} precision={precision:.3f} recall={recall:.3f} f1={f1:.3f}")

  acc = sum(1 for p, g in zip(preds, gold) if p == g) / len(gold)
  print(f"\nOverall accuracy: {acc:.3f}")


classification_report(preds, gold, idx2label)

Child Sexual Exploitation precision=1.000 recall=1.000 f1=1.000
Elections            precision=1.000 recall=1.000 f1=1.000
Non-Violent Crimes   precision=0.000 recall=0.000 f1=0.000
Safe                 precision=0.395 recall=0.986 f1=0.564
Sex-Related Crimes   precision=0.917 recall=1.000 f1=0.957
Suicide & Self-Harm  precision=1.000 recall=0.950 f1=0.974
Unknown S-Type       precision=1.000 recall=0.516 f1=0.681
Violent Crimes       precision=0.000 recall=0.000 f1=0.000
unsafe               precision=0.000 recall=0.000 f1=0.000

Overall accuracy: 0.493
